# Football tracking + ball possession

Detects and tracks players and the ball in a football video, splits the players into two teams
automatically (from the jersey colors, no configuration), decides who has the ball and writes a video with
the tracking boxes and a live possession board. At the end it prints the possession percentage per team.

Built on top of [tryolabs/soccer-video-analytics](https://github.com/tryolabs/soccer-video-analytics) (MIT license).

**How to use**
1. `Runtime > Change runtime type > GPU` (T4 is enough).
2. Run the cells in order, **steps 1 to 6 only once per session**.
3. Steps 7 and 8 are the ones you repeat: run them to **choose a video** (a *Choose files* button appears) and process it.

> If Colab shows a *Restart session* message after step 3, click it and continue from step 4.

## 1. Check the GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi -L

## 2. Mount Google Drive (where your model weights live)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the base repository and install the dependencies

`norfair` pins `numpy<2`, which breaks Colab's numpy, so it is installed without its dependencies.

In [ ]:
%cd /content
!git clone https://github.com/tryolabs/soccer-video-analytics.git
%cd /content/soccer-video-analytics

!pip install -q norfair --no-deps
!pip install -q filterpy ultralytics

## 4. Compatibility patches for current Colab (Pillow >= 10, NumPy 2) and detector settings

In [ ]:
%cd /content/soccer-video-analytics

# Pillow >= 10 removed ImageDraw.textsize, NumPy 2 removed np.round_
!sed -i 's/w, h = draw.textsize(text, font=font)/_l, _t, _r, _b = draw.textbbox((0, 0), text, font=font); w, h = _r - _l, _b - _t/' soccer/draw.py
!sed -i 's/np\.round_(/np.round(/g' soccer/ball.py

# accept lower-confidence ball detections (the repo default of 0.3 drops too many frames)
u = open("run_utils.py").read()
u = u.replace('ball_df = ball_df[ball_df["confidence"] > 0.3]', 'ball_df = ball_df[ball_df["confidence"] > 0.15]')
open("run_utils.py", "w").write(u)

# trust the YOLOv5 hub repo once, so torch.hub never stops to ask "trust this repository?"
import torch
try:
    torch.hub.list("ultralytics/yolov5", trust_repo=True)
except Exception as e:
    print("Could not pre-trust the yolov5 hub repo (answer 'y' if it asks later):", e)

## 5. Project files

These cells write the project's own code next to the base repository. Nothing to edit here.

| file | what it does |
|---|---|
| `auto_team_classifier.py` | splits players into two teams from jersey colors (k-means), referee/goalkeepers become "Other" |
| `ultralytics_detector.py` | lets the pipeline use models trained with `ultralytics` (YOLOv8/11) |
| `custom_board.py` | the possession board drawn on the video |
| `viz_tweaks.py` | high quality H.264 output, upscaling of small videos, thinner player boxes |
| `run.py` | the pipeline itself |
| `compute_xg.py` | detects shots from the ball's speed/direction and estimates a heuristic xG per shot (run after `run.py`) |
| `xg_overlay.py` | draws the shots + a running xG total onto the video (run after `compute_xg.py`) |

### `auto_team_classifier.py`

In [ ]:
%%writefile /content/soccer-video-analytics/auto_team_classifier.py
"""
AutoTeamClassifier: splits players into two teams automatically, with no jersey
colors to configure. Drop-in replacement for HSVClassifier in
tryolabs/soccer-video-analytics (put this file next to run.py).

How it works
------------
1. Warm-up: a few dozen frames are sampled from the video, the player detector
   is run on them, and for every player we take the median color (Lab space) of
   the torso, ignoring grass-colored pixels.
2. k-means on those colors; the two biggest clusters are the two teams. Points that are far from
   both clusters (referee, goalkeepers in a different kit, spectators) are
   trimmed out and labeled "Other" when predicting, so they are never counted
   for either team.
3. During the run, every tracked player is assigned to the nearest team by
   jersey color. InertiaClassifier (already used by run.py) then smooths the
   label per tracker ID over the last frames.
"""
from typing import Callable, List, Optional, Sequence, Tuple

import cv2
import numpy as np

from inference.base_classifier import BaseClassifier
from inference.box import Box

OTHER = "Other"


class AutoTeamClassifier(BaseClassifier):
    def __init__(
        self,
        team_names: Sequence[str] = ("Team A", "Team B"),
        l_weight: float = 0.5,
        outlier_sigma: float = 3.0,
        min_threshold: float = 15.0,
        seed: int = 0,
    ):
        """
        team_names    : names of the two teams (must match the Team(...) names in run.py)
        l_weight      : weight of lightness vs. color in the distance (lower = less
                        sensitive to shadows / sun)
        outlier_sigma : how far (in robust std devs) from a team center a player can be
                        before being labeled "Other"
        min_threshold : lower bound for that distance, in the same units
        """
        if len(team_names) != 2:
            raise ValueError("team_names must have exactly 2 names")
        self.team_names = list(team_names)
        self.l_weight = l_weight
        self.outlier_sigma = outlier_sigma
        self.min_threshold = min_threshold
        self.seed = seed

        self.centroids: Optional[np.ndarray] = None  # (2, 3)
        self.thresholds: Optional[np.ndarray] = None  # (2,)
        self.colors_bgr: Optional[np.ndarray] = None  # (2, 3)

    # ------------------------------------------------------------------ features
    def features(self, img: np.ndarray) -> Optional[Tuple[np.ndarray, np.ndarray]]:
        """(weighted Lab feature, median BGR color) of the jersey area, or None."""
        if img is None or img.ndim != 3 or img.shape[0] < 12 or img.shape[1] < 6:
            return None

        h, w = img.shape[:2]
        crop = img[int(h * 0.15) : int(h * 0.55), int(w * 0.2) : int(w * 0.8)]
        if crop.size == 0:
            return None
        crop = np.ascontiguousarray(crop)

        pix = crop.reshape(-1, 3)
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV).reshape(-1, 3)
        grass = (hsv[:, 0] >= 35) & (hsv[:, 0] <= 85) & (hsv[:, 1] > 40) & (hsv[:, 2] > 40)
        keep = ~grass
        # if (almost) everything is green, the kit itself is green: use all pixels
        pix_use = pix[keep] if keep.mean() >= 0.25 else pix

        lab = cv2.cvtColor(np.ascontiguousarray(pix_use.reshape(-1, 1, 3)), cv2.COLOR_BGR2LAB)
        lab = lab.reshape(-1, 3).astype(np.float32)
        feat = np.median(lab, axis=0) * np.array([self.l_weight, 1.0, 1.0], np.float32)
        bgr = np.median(pix_use, axis=0).astype(np.float32)
        return feat, bgr

    # ---------------------------------------------------------------------- fit
    def fit_features(self, feats: List[np.ndarray], bgrs: List[np.ndarray]) -> dict:
        X = np.asarray(feats, np.float32)
        B = np.asarray(bgrs, np.float32)
        if len(X) < 8:
            raise ValueError(
                f"Only {len(X)} player crops found in the warm-up frames; "
                "need at least 8 to separate the teams."
            )

        cv2.setRNGSeed(self.seed)
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.01)
        # Over-cluster (k=4) so referee / goalkeepers get their own small clusters instead
        # of stealing one of the two team clusters; the two BIGGEST clusters are the teams.
        k = 4 if len(X) >= 60 else 2
        _, labels0, centers0 = cv2.kmeans(X, k, None, criteria, 10, cv2.KMEANS_PP_CENTERS)
        counts = np.bincount(labels0.ravel(), minlength=k)
        centers = centers0[np.argsort(-counts)[:2]].copy()

        # trimmed refinement: recompute centers from inliers only
        for _ in range(3):
            d_all = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
            labels = d_all.argmin(axis=1)
            thr = np.zeros(2, np.float32)
            inlier = np.zeros(len(X), bool)
            for k in range(2):
                dk = d_all[labels == k, k]
                if len(dk) == 0:
                    raise ValueError("One of the two team clusters is empty.")
                med = np.median(dk)
                mad = np.median(np.abs(dk - med)) * 1.4826
                thr[k] = max(med + self.outlier_sigma * max(mad, 4.0), self.min_threshold)
                inlier |= (labels == k) & (d_all[:, k] <= thr[k])
            for k in range(2):
                members = X[inlier & (labels == k)]
                if len(members):
                    centers[k] = np.median(members, axis=0)

        d_all = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
        labels = d_all.argmin(axis=1)
        colors = np.zeros((2, 3), np.float32)
        sizes = []
        for k in range(2):
            members = inlier & (labels == k)
            sizes.append(int(members.sum()))
            colors[k] = np.median(B[members], axis=0) if members.any() else B[labels == k].mean(0)

        self.centroids = centers.astype(np.float32)
        self.thresholds = thr
        self.colors_bgr = colors

        separation = float(np.linalg.norm(centers[0] - centers[1]))
        return {
            "samples": int(len(X)),
            "team_sizes": sizes,
            "outliers": int(len(X) - inlier.sum()),
            "separation": separation,
            "thresholds": [float(t) for t in thr],
        }

    def fit_from_video(
        self,
        video_path: str,
        detect_fn: Callable[[np.ndarray], list],
        n_frames: int = 40,
        min_box: Tuple[int, int] = (8, 24),
        verbose: bool = True,
    ) -> dict:
        """
        Sample ~n_frames frames evenly from the video, run detect_fn(frame) (must return
        norfair Detections whose points are [[x1, y1], [x2, y2]]) and fit the two teams.
        """
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Could not open video: {video_path}")
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        stride = max(total // max(n_frames, 1), 1)

        feats, bgrs = [], []
        i = 0
        while True:
            if i % stride == 0:
                ok, frame = cap.read()
            else:
                ok, frame = cap.grab(), None
            if not ok:
                break
            if frame is not None:
                fh, fw = frame.shape[:2]
                for det in detect_fn(frame):
                    (x1, y1), (x2, y2) = det.points[0], det.points[1]
                    x1, x2 = max(int(x1), 0), min(int(x2), fw)
                    y1, y2 = max(int(y1), 0), min(int(y2), fh)
                    if (x2 - x1) < min_box[0] or (y2 - y1) < min_box[1]:
                        continue
                    r = self.features(Box((x1, y1), (x2, y2), frame).img)
                    if r is not None:
                        feats.append(r[0])
                        bgrs.append(r[1])
            i += 1
        cap.release()

        info = self.fit_features(feats, bgrs)
        if verbose:
            print(
                f"[AutoTeamClassifier] {info['samples']} player crops from warm-up | "
                f"team sizes {info['team_sizes']} | outliers (referee/GK/other) {info['outliers']} | "
                f"team separation {info['separation']:.0f}"
            )
            if info["separation"] < 25:
                print(
                    "[AutoTeamClassifier] WARNING: the two teams' jersey colors look very similar; "
                    "the split may be unreliable."
                )
        return info

    def team_colors(self) -> List[Tuple[int, int, int]]:
        """Median jersey color per team as (B, G, R) ints, same order as team_names."""
        if self.colors_bgr is None:
            raise RuntimeError("Call fit_from_video() first.")
        return [tuple(int(v) for v in c) for c in self.colors_bgr]

    # ------------------------------------------------------------------ predict
    def _predict_one(self, img: np.ndarray) -> str:
        r = self.features(img)
        if r is None:
            return OTHER
        d = np.linalg.norm(self.centroids - r[0], axis=1)
        k = int(d.argmin())
        return self.team_names[k] if d[k] <= self.thresholds[k] else OTHER

    def predict(self, input_image: List[np.ndarray]) -> List[str]:
        if self.centroids is None:
            raise RuntimeError("Call fit_from_video() before predicting.")
        if not isinstance(input_image, list):
            input_image = [input_image]
        return [self._predict_one(img) for img in input_image]

### `ultralytics_detector.py`

In [ ]:
%%writefile /content/soccer-video-analytics/ultralytics_detector.py
"""
UltralyticsDetector: same interface as the repo's YoloV5 detector
(predict(frame) -> DataFrame with xmin, ymin, xmax, ymax, confidence, class, name),
but loads a model trained with the `ultralytics` package (YOLOv8 / YOLO11).

The repo's YoloV5 class can only load YOLOv5-format weights, so it fails on a
YOLOv8 model with: "'Detect' object has no attribute 'grid'".
"""
import numpy as np
import pandas as pd

from inference.base_detector import BaseDetector

COLUMNS = ["xmin", "ymin", "xmax", "ymax", "confidence", "class", "name"]


class UltralyticsDetector(BaseDetector):
    def __init__(self, model_path: str, conf: float = 0.25, imgsz: int = 640):
        """
        model_path : path to the .pt trained with ultralytics
        conf       : minimum confidence returned by the model (run_utils filters again at 0.3)
        imgsz      : inference size; raise it (e.g. 960 / 1280) if the ball is small in your video
        """
        from ultralytics import YOLO

        self.model = YOLO(model_path)
        self.conf = conf
        self.imgsz = imgsz
        print(f"[UltralyticsDetector] loaded {model_path} | classes: {self.model.names}")

    def predict(self, input_image: np.ndarray) -> pd.DataFrame:
        # numpy frames are treated as BGR (cv2 convention), which is what the video gives us
        result = self.model.predict(
            input_image, conf=self.conf, imgsz=self.imgsz, verbose=False
        )[0]

        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            return pd.DataFrame(columns=COLUMNS)

        xyxy = boxes.xyxy.cpu().numpy()
        conf = boxes.conf.cpu().numpy()
        cls = boxes.cls.cpu().numpy().astype(int)

        return pd.DataFrame(
            {
                "xmin": xyxy[:, 0],
                "ymin": xyxy[:, 1],
                "xmax": xyxy[:, 2],
                "ymax": xyxy[:, 3],
                "confidence": conf,
                "class": cls,
                "name": [result.names[int(c)] for c in cls],
            }
        )

### `custom_board.py`

In [ ]:
%%writefile /content/soccer-video-analytics/custom_board.py
"""
Custom possession board for tryolabs/soccer-video-analytics (put next to run.py).

Same information as the repo's board (team names, possession time, possession %),
different look: a compact dark panel at the top-right with team chips, big percentages,
possession time in the middle and a split bar. A small amber dot marks the team that
currently has the ball.

Every visual choice is a constant at the top of the file, so it is easy to tweak.
Colors here are written in BGR order because the repo hands PIL a BGR frame (that is why
ACCENT = (0, 190, 255) shows up as amber in the video).
"""
from functools import lru_cache

import PIL.Image
import PIL.ImageDraw
import PIL.ImageFont

FONT_PATH = "fonts/Gidole-Regular.ttf"  # comes with the repo; run from the repo root

# ---- look & feel (base size: 1280 px wide frame, everything scales with the frame width) ----
PANEL_SIZE = (430, 104)
PANEL_RADIUS = 16
PASSES_PANEL_SIZE = (430, 74)
PASSES_GAP = 14  # gap between the bottom of the possession board and the top of the passes board
PANEL_BG = (20, 20, 24, 205)  # RGBA, last number = opacity
PANEL_BORDER = (255, 255, 255, 45)
TRACK = (60, 60, 66, 255)
TEXT_MAIN = (255, 255, 255, 255)
TEXT_DIM = (176, 176, 184, 255)
ACCENT = (0, 190, 255, 255)  # "has the ball" dot (BGR -> amber)
TITLE = "BALL POSSESSION"
MARGIN = 30  # distance from the top / right edge
SS = 2  # supersampling, for smooth edges


@lru_cache(maxsize=32)
def _font(size: int):
    try:
        return PIL.ImageFont.truetype(FONT_PATH, size=size)
    except OSError:
        try:
            return PIL.ImageFont.load_default(size=size)
        except TypeError:
            return PIL.ImageFont.load_default()


def _text(draw, xy, text, size, fill, anchor="l"):
    """Draw text with its vertical middle at xy[1]; anchor = l / m / r for the x position."""
    font = _font(size)
    l, t, r, b = draw.textbbox((0, 0), text, font=font)
    w, h = r - l, b - t
    x, y = xy
    if anchor == "m":
        x -= w / 2
    elif anchor == "r":
        x -= w
    draw.text((x - l, y - h / 2 - t), text, font=font, fill=fill)


def _rgb(color):
    return tuple(int(c) for c in color[:3])


def draw_possession_board(frame: PIL.Image.Image, match, origin=None, scale=None):
    """
    frame  : PIL image (the same one run.py draws on)
    match  : soccer.Match
    origin : (x, y) of the panel's top-left corner; default = top-right
    scale  : override the automatic scale (frame_width / 1280)
    """
    W, H = frame.size
    s = scale if scale else W / 1280.0
    pw, ph = int(round(PANEL_SIZE[0] * s)), int(round(PANEL_SIZE[1] * s))
    margin = int(round(MARGIN * s))
    x0, y0 = origin if origin else (W - pw - margin, margin)

    k = s * SS

    def u(v):  # base units -> supersampled pixels
        return int(round(v * k))

    home, away = match.home, match.away
    home_c, away_c = _rgb(home.board_color) + (255,), _rgb(away.board_color) + (255,)

    # possession split
    if match.duration > 0:
        ratio = home.get_percentage_possession(match.duration)
    else:
        ratio = 0.5
    home_pct = int(round(ratio * 100))
    away_pct = 100 - home_pct
    ratio = min(max(ratio, 0.04), 0.96)  # keep both bar segments visible

    W0, H0 = PANEL_SIZE
    img = PIL.Image.new("RGBA", (pw * SS, ph * SS), (0, 0, 0, 0))
    d = PIL.ImageDraw.Draw(img)

    # panel
    d.rounded_rectangle(
        [0, 0, pw * SS - 1, ph * SS - 1],
        radius=u(PANEL_RADIUS),
        fill=PANEL_BG,
        outline=PANEL_BORDER,
        width=SS,
    )

    # team chips + title
    chip_w, chip_h, chip_y = 66, 26, 12
    for team, color, x in ((home, home_c, 16), (away, away_c, W0 - 16 - chip_w)):
        d.rounded_rectangle(
            [u(x), u(chip_y), u(x + chip_w), u(chip_y + chip_h)],
            radius=u(chip_h / 2),
            fill=color,
        )
        _text(d, (u(x + chip_w / 2), u(chip_y + chip_h / 2)), team.abbreviation,
              u(15), _rgb(team.text_color) + (255,), "m")
    _text(d, (u(W0 / 2), u(chip_y + chip_h / 2)), TITLE, u(12), TEXT_DIM, "m")

    # "has the ball" dot
    dot_y, dot_r = chip_y + chip_h / 2, 4
    holder = match.team_possession
    if holder is not None:
        dot_x = 16 + chip_w + 12 if holder is home else W0 - 16 - chip_w - 12
        d.ellipse([u(dot_x - dot_r), u(dot_y - dot_r), u(dot_x + dot_r), u(dot_y + dot_r)], fill=ACCENT)

    # percentages + possession time
    row_y = 55
    _text(d, (u(16), u(row_y)), f"{home_pct}%", u(32), TEXT_MAIN, "l")
    _text(d, (u(W0 - 16), u(row_y)), f"{away_pct}%", u(32), TEXT_MAIN, "r")
    times = f"{home.get_time_possession(match.fps)}  -  {away.get_time_possession(match.fps)}"
    _text(d, (u(W0 / 2), u(row_y)), times, u(14), TEXT_DIM, "m")

    # split bar
    bx0, bx1, by0, by1 = 16, W0 - 16, 80, 92
    split = bx0 + ratio * (bx1 - bx0)
    r = (by1 - by0) / 2
    d.rounded_rectangle([u(bx0), u(by0), u(bx1), u(by1)], radius=u(r), fill=TRACK)
    d.rounded_rectangle([u(bx0), u(by0), u(split - 2), u(by1)], radius=u(r), fill=home_c)
    d.rounded_rectangle([u(split + 2), u(by0), u(bx1), u(by1)], radius=u(r), fill=away_c)

    overlay = img.resize((pw, ph), PIL.Image.LANCZOS)

    box = (x0, y0, x0 + pw, y0 + ph)
    region = frame.crop(box).convert("RGBA")
    region = PIL.Image.alpha_composite(region, overlay)
    frame.paste(region.convert(frame.mode), (x0, y0))
    return frame


def draw_passes_board(frame: PIL.Image.Image, match, origin=None, scale=None):
    """
    Small panel with each team's pass count, meant to sit directly under draw_possession_board.
    Pass counting itself is the repo's own logic (soccer/pass_event.py): a pass is counted
    whenever the ball moves from one player to a different player on the same team.
    """
    W, H = frame.size
    s = scale if scale else W / 1280.0
    pw, ph = int(round(PASSES_PANEL_SIZE[0] * s)), int(round(PASSES_PANEL_SIZE[1] * s))
    margin = int(round(MARGIN * s))
    if origin:
        x0, y0 = origin
    else:
        pos_h = int(round(PANEL_SIZE[1] * s))
        x0 = W - pw - margin
        y0 = margin + pos_h + int(round(PASSES_GAP * s))

    k = s * SS

    def u(v):
        return int(round(v * k))

    home, away = match.home, match.away
    home_c, away_c = _rgb(home.board_color) + (255,), _rgb(away.board_color) + (255,)
    home_n, away_n = len(home.passes), len(away.passes)
    total = home_n + away_n
    ratio = home_n / total if total else 0.5
    ratio = min(max(ratio, 0.04), 0.96) if total else 0.5

    W0, H0 = PASSES_PANEL_SIZE
    img = PIL.Image.new("RGBA", (pw * SS, ph * SS), (0, 0, 0, 0))
    d = PIL.ImageDraw.Draw(img)

    d.rounded_rectangle(
        [0, 0, pw * SS - 1, ph * SS - 1],
        radius=u(PANEL_RADIUS), fill=PANEL_BG, outline=PANEL_BORDER, width=SS,
    )

    row_y = 30
    _text(d, (u(16), u(row_y)), str(home_n), u(26), home_c, "l")
    _text(d, (u(W0 - 16), u(row_y)), str(away_n), u(26), away_c, "r")
    _text(d, (u(W0 / 2), u(row_y - 12)), "PASSES", u(11), TEXT_DIM, "m")
    _text(d, (u(W0 / 2), u(row_y + 12)), f"{home.abbreviation}  -  {away.abbreviation}", u(11), TEXT_DIM, "m")

    bx0, bx1, by0, by1 = 16, W0 - 16, 58, 66
    r = (by1 - by0) / 2
    d.rounded_rectangle([u(bx0), u(by0), u(bx1), u(by1)], radius=u(r), fill=TRACK)
    if total:
        split = bx0 + ratio * (bx1 - bx0)
        d.rounded_rectangle([u(bx0), u(by0), u(split - 2), u(by1)], radius=u(r), fill=home_c)
        d.rounded_rectangle([u(split + 2), u(by0), u(bx1), u(by1)], radius=u(r), fill=away_c)

    overlay = img.resize((pw, ph), PIL.Image.LANCZOS)
    box = (x0, y0, x0 + pw, y0 + ph)
    region = frame.crop(box).convert("RGBA")
    region = PIL.Image.alpha_composite(region, overlay)
    frame.paste(region.convert(frame.mode), (x0, y0))
    return frame

### `viz_tweaks.py`

In [ ]:
%%writefile /content/soccer-video-analytics/viz_tweaks.py
"""
viz_tweaks.py - three visual fixes for run.py (put this file next to run.py):

1. HQWriter      : writes the output with ffmpeg / H.264 at high quality. norfair's own writer
                   uses the old "mp4v" codec at a low bitrate, which is what makes the result look blocky.
2. prepare_frame : if the video is narrower than 1280 px, upscale it (bicubic) so the repo's overlays
                   fit; frames that are already >= 1280 px wide are left untouched.
3. thin_boxes    : thinner outline for the player boxes (repo default is 3 px).
"""
import shutil
import subprocess
from functools import lru_cache

import PIL.Image  # noqa: F401  (soccer.draw needs PIL.Image to be loaded first)
import PIL.ImageDraw  # noqa: F401
import PIL.ImageFont  # noqa: F401
import cv2
import numpy as np

from soccer.draw import Draw

# ---- settings you may want to tweak ----------------------------------------
BOX_THICKNESS = 2   # player box line width in px (repo default: 3, use 1 for very thin)
CRF = 16            # H.264 quality: lower = better + bigger file (14 ~ near lossless, 18 = good)
PRESET = "slow"     # slower = a bit smaller file at the same quality
MIN_WIDTH = 1280    # upscale videos narrower than this
GOAL_COLOR = (255, 220, 0)  # goal boxes; BGR order, shows up as cyan in the video
GOAL_WIDTH = 2
GOAL_FONT = "fonts/Gidole-Regular.ttf"  # comes with the repo; run from the repo root
# ------------------------------------------------------------------------------


def thin_boxes(thickness: int = BOX_THICKNESS):
    """Make Draw.draw_bounding_box (used for every player box) use a thinner line."""
    if getattr(Draw.draw_bounding_box, "_thin", False):
        return
    original = Draw.draw_bounding_box

    def thin(img, rectangle, color, thickness_arg=None, **kwargs):
        return original(img=img, rectangle=rectangle, color=color, thickness=thickness)

    thin._thin = True
    Draw.draw_bounding_box = staticmethod(thin)


def prepare_frame(frame: np.ndarray) -> np.ndarray:
    h, w = frame.shape[:2]
    if w >= MIN_WIDTH:
        return frame
    new_w = MIN_WIDTH
    new_h = int(round(h * MIN_WIDTH / w / 2)) * 2  # keep the height even (H.264 needs it)
    return cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_CUBIC)


class HQWriter:
    def __init__(self, video, fps: float, crf: int = CRF, preset: str = PRESET):
        """video: the norfair Video (used only for the output file name / as fallback)."""
        self.video = video
        self.fps = fps if fps and fps > 0 else 30.0
        self.crf = crf
        self.preset = preset
        self.path = video.get_output_file_path()
        self.proc = None
        self.use_ffmpeg = shutil.which("ffmpeg") is not None
        if not self.use_ffmpeg:
            print("[HQWriter] ffmpeg not found: falling back to norfair's writer (lower quality).")

    def write(self, frame: np.ndarray):
        if not self.use_ffmpeg:
            self.video.write(frame)
            return
        if self.proc is None:
            h, w = frame.shape[:2]
            cmd = [
                "ffmpeg", "-y", "-loglevel", "error",
                "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{w}x{h}", "-r", f"{self.fps}",
                "-i", "-",
                "-an",
                "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
                "-c:v", "libx264", "-preset", self.preset, "-crf", str(self.crf),
                "-pix_fmt", "yuv420p", "-movflags", "+faststart",
                self.path,
            ]
            self.proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
        try:
            self.proc.stdin.write(np.ascontiguousarray(frame).tobytes())
        except BrokenPipeError:
            raise RuntimeError("ffmpeg stopped unexpectedly while writing the video.")

    def close(self):
        if self.proc is not None:
            self.proc.stdin.close()
            self.proc.wait()
            print(f"Output video file saved to: {self.path}")
            self.proc = None


@lru_cache(maxsize=4)
def _goal_font(size: int):
    try:
        return PIL.ImageFont.truetype(GOAL_FONT, size=size)
    except OSError:
        return PIL.ImageFont.load_default()


def draw_goal_boxes(frame, goal_df, size: int = 15):
    """Draw the detections of the goal model (thin cyan box + name and confidence)."""
    if goal_df is None or len(goal_df) == 0:
        return frame
    draw = PIL.ImageDraw.Draw(frame)
    font = _goal_font(size)
    for r in goal_df.to_dict("records"):
        x1, y1, x2, y2 = r["xmin"], r["ymin"], r["xmax"], r["ymax"]
        draw.rectangle([x1, y1, x2, y2], outline=GOAL_COLOR, width=GOAL_WIDTH)
        draw.text((x1 + 3, max(y1 - size - 4, 0)), f'{r["name"]} {r["confidence"]:.2f}', fill=GOAL_COLOR, font=font)
    return frame

### `run.py`

In [ ]:
%%writefile /content/soccer-video-analytics/run.py
"""
Football possession + tracking on top of tryolabs/soccer-video-analytics.

  python run.py --video videos/match.mp4 --model models/ball.pt
  python run.py --video videos/match.mp4 --model models/ball.pt --player-model models/player.pt

What it does per frame: detect players + ball, track them (norfair), split the players into
two teams automatically (jersey colors), decide who has the ball, and write a video with the
tracking boxes and a possession board. The final possession percentages are printed at the end.
"""
import argparse
import csv
import json
import os

import cv2

# Colab / servers have no GUI: norfair calls these after writing frames
cv2.waitKey = lambda *a, **k: -1
cv2.destroyAllWindows = lambda *a, **k: None

import numpy as np
import PIL
import PIL.Image
from norfair import Tracker, Video
from norfair.camera_motion import MotionEstimator
from norfair.distances import mean_euclidean

from auto_team_classifier import AutoTeamClassifier
from custom_board import draw_passes_board, draw_possession_board
from inference import Converter, InertiaClassifier, YoloV5
from run_utils import (
    get_ball_detections,
    get_main_ball,
    get_player_detections,
    update_motion_estimator,
)
from soccer import Match, Player, Team
from soccer.draw import AbsolutePath
from ultralytics_detector import UltralyticsDetector
from viz_tweaks import HQWriter, draw_goal_boxes, prepare_frame, thin_boxes

parser = argparse.ArgumentParser()
parser.add_argument("--video", default="videos/soccer_possession.mp4", type=str, help="Input video")
parser.add_argument("--model", default="models/ball.pt", type=str, help="Ball detector (ultralytics .pt)")
parser.add_argument("--out-dir", default="outputs", type=str, help="Folder for the output video")
parser.add_argument(
    "--possession",
    action=argparse.BooleanOptionalAction,
    default=True,
    help="Draw tracking + possession board (on by default)",
)
parser.add_argument(
    "--passes",
    action=argparse.BooleanOptionalAction,
    default=True,
    help="Draw the passes count board under the possession board (on by default)",
)
parser.add_argument("--warmup-frames", default=40, type=int, help="Frames sampled to learn the team colors")
parser.add_argument("--ball-imgsz", default=1280, type=int, help="Inference size of the ball model")
parser.add_argument("--ball-conf", default=0.10, type=float, help="Minimum confidence of the ball model")
parser.add_argument(
    "--player-model",
    default=None,
    type=str,
    help="Optional: your own player detector (ultralytics .pt). Default: pretrained YOLOv5x (COCO 'person')",
)
parser.add_argument(
    "--player-classes",
    default=None,
    type=str,
    help="Comma-separated class names to keep from --player-model (default: every class except the ball)",
)
parser.add_argument("--player-conf", default=0.35, type=float, help="Minimum confidence for --player-model")
parser.add_argument("--player-imgsz", default=1280, type=int, help="Inference size for --player-model")
parser.add_argument(
    "--goal-model",
    default=None,
    type=str,
    help="Optional: goal detector (ultralytics .pt). Draws the goal boxes and saves them to a CSV",
)
parser.add_argument("--goal-conf", default=0.30, type=float, help="Minimum confidence for --goal-model")
parser.add_argument("--goal-imgsz", default=1280, type=int, help="Inference size for --goal-model")
args = parser.parse_args()

os.makedirs(args.out_dir, exist_ok=True)
video = Video(input_path=args.video, output_path=args.out_dir)
fps = video.video_capture.get(cv2.CAP_PROP_FPS)
thin_boxes()
writer = HQWriter(video, fps)

# Object detectors
if args.player_model:
    player_detector = UltralyticsDetector(
        model_path=args.player_model, conf=args.player_conf, imgsz=args.player_imgsz
    )
    keep_classes = (
        {c.strip().lower() for c in args.player_classes.split(",") if c.strip()}
        if args.player_classes
        else None
    )

    def detect_players(frame):
        df = player_detector.predict(frame)
        names = df["name"].astype(str).str.lower()
        df = df[names.isin(keep_classes)] if keep_classes else df[~names.str.contains("ball")]
        return Converter.DataFrame_to_Detections(df)

else:
    player_detector = YoloV5()

    def detect_players(frame):
        return get_player_detections(player_detector, frame)


ball_detector = UltralyticsDetector(
    model_path=args.model, imgsz=args.ball_imgsz, conf=args.ball_conf
)

# Optional goal detector (for the goal boxes on the video and the CSV used later for xG work)
goal_detector = None
goal_rows = []
ball_rows = []  # per-frame ball position + team in possession, used later for shot/xG detection
if args.goal_model:
    goal_detector = UltralyticsDetector(
        model_path=args.goal_model, conf=args.goal_conf, imgsz=args.goal_imgsz
    )

# Automatic team classifier: learns the 2 jersey colors from the video itself
auto_classifier = AutoTeamClassifier(team_names=["Team A", "Team B"])
auto_classifier.fit_from_video(args.video, detect_players, n_frames=args.warmup_frames)

# Add inertia to classifier
classifier = InertiaClassifier(classifier=auto_classifier, inertia=20)


def text_color_for(bgr):
    luminance = 0.114 * bgr[0] + 0.587 * bgr[1] + 0.299 * bgr[2]
    return (0, 0, 0) if luminance > 140 else (255, 255, 255)


# Teams and Match (colors = the real jersey colors found in the video)
color_a, color_b = auto_classifier.team_colors()
team_a = Team(
    name="Team A",
    abbreviation="TMA",
    color=color_a,
    board_color=color_a,
    text_color=text_color_for(color_a),
)
team_b = Team(
    name="Team B",
    abbreviation="TMB",
    color=color_b,
    board_color=color_b,
    text_color=text_color_for(color_b),
)
teams = [team_a, team_b]
match = Match(home=team_a, away=team_b, fps=fps)

# Tracking
player_tracker = Tracker(
    distance_function=mean_euclidean,
    distance_threshold=250,
    initialization_delay=3,
    hit_counter_max=90,
)

ball_tracker = Tracker(
    distance_function=mean_euclidean,
    distance_threshold=150,
    initialization_delay=5,
    hit_counter_max=2000,
)
motion_estimator = MotionEstimator()
coord_transformations = None

# Paths
path = AbsolutePath()

# Get Counter img
for i, frame in enumerate(video):
    frame = prepare_frame(frame)

    # Goal detections (optional)
    goal_df = None
    if goal_detector is not None:
        goal_df = goal_detector.predict(frame)
        for r in goal_df.to_dict("records"):
            goal_rows.append(
                [i, r["name"], round(float(r["confidence"]), 4)]
                + [round(float(r[k]), 1) for k in ("xmin", "ymin", "xmax", "ymax")]
                + [frame.shape[1], frame.shape[0]]
            )

    # Get Detections
    players_detections = detect_players(frame)
    ball_detections = get_ball_detections(ball_detector, frame)
    detections = ball_detections + players_detections

    # Update trackers
    coord_transformations = update_motion_estimator(
        motion_estimator=motion_estimator,
        detections=detections,
        frame=frame,
    )

    player_track_objects = player_tracker.update(
        detections=players_detections, coord_transformations=coord_transformations
    )

    ball_track_objects = ball_tracker.update(
        detections=ball_detections, coord_transformations=coord_transformations
    )

    player_detections = Converter.TrackedObjects_to_Detections(player_track_objects)
    ball_detections = Converter.TrackedObjects_to_Detections(ball_track_objects)

    player_detections = classifier.predict_from_detections(
        detections=player_detections,
        img=frame,
    )

    # Match update
    ball = get_main_ball(ball_detections)
    players = Player.from_detections(detections=players_detections, teams=teams)
    match.update(players, ball)

    center = ball.center if ball else None  # (x, y) in the (possibly resized) frame, or None
    bx, by = (float(center[0]), float(center[1])) if center is not None else ("", "")
    ball_rows.append([i, bx, by, match.team_possession.name if match.team_possession else ""])

    # Draw
    frame = PIL.Image.fromarray(frame)
    frame = draw_goal_boxes(frame, goal_df)

    if args.possession:
        frame = Player.draw_players(
            players=players, frame=frame, confidence=False, id=True
        )

        frame = path.draw(
            img=frame,
            detection=ball.detection,
            coord_transformations=coord_transformations,
            color=match.team_possession.color,
        )

        frame = draw_possession_board(frame, match)
        if args.passes:
            frame = draw_passes_board(frame, match)
        if match.closest_player:
            frame = match.closest_player.draw_pointer(frame)

        if ball:
            frame = ball.draw(frame)

    frame = np.array(frame)

    # Write video
    writer.write(frame)

writer.close()

stem = os.path.basename(args.video).split(".")[0]

teams_json = os.path.join(args.out_dir, f"{stem}_teams.json")
with open(teams_json, "w") as f:
    json.dump(
        {t.name: {"color_bgr": list(t.color), "abbreviation": t.abbreviation} for t in teams},
        f, indent=2,
    )

ball_csv = os.path.join(args.out_dir, f"{stem}_ball.csv")
with open(ball_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["frame", "x", "y", "team_possession"])
    w.writerows(ball_rows)
print(f"Ball positions: {len(ball_rows)} rows saved to {ball_csv}")

if goal_detector is not None:
    goal_csv = os.path.join(args.out_dir, f"{stem}_goals.csv")
    with open(goal_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["frame", "name", "confidence", "x1", "y1", "x2", "y2", "frame_w", "frame_h"])
        w.writerows(goal_rows)
    print(f"Goal detections: {len(goal_rows)} rows saved to {goal_csv}")
else:
    print("No --goal-model given: skipping goal detections and xG (the ball CSV above is still saved).")

# Final possession summary
print("\n===== Possession =====")
for team in teams:
    print(f"{team.name}: {team.get_percentage_possession(match.duration):.0%}")

### `compute_xg.py`

In [ ]:
%%writefile /content/soccer-video-analytics/compute_xg.py
"""
compute_xg.py - shot detection and a simple geometric xG, from the CSVs run.py already saves
(no shot-outcome dataset needed, and nothing to train).

    python compute_xg.py --ball outputs/match_ball.csv --goals outputs/match_goals.csv --fps 30

What it does
------------
1. Shot detection: looks at the ball's frame-to-frame speed and direction. A "shot" is a burst
   of frames where the ball moves fast and roughly straight at a goal. Nearby bursts are merged
   into one shot; the shot's location is the ball position where the burst starts (i.e. roughly
   where it was kicked from), not where it ends up.
2. xG: uses the goal box the goal model already found (its pixel width stands in for the real
   goal width, 7.32 m, so pixel distances near the goal can be turned into meters) to get the
   shot's distance and shooting angle to goal, then a hand-set logistic curve turns those into a
   probability.

IMPORTANT — this is a heuristic, not a statistical model: the curve's coefficients were picked by
hand to look reasonable (closer + straighter-on = higher), not fitted to real shot outcomes. Two
approximations to keep in mind: the meters-per-pixel scale is derived from the goal box, which is
least accurate for shots taken from far away or from a sharp angle; and it doesn't know about
defenders, the goalkeeper's position, or whether the shot was blocked. Treat the numbers as
"relative shot quality on this clip", not literal goal probabilities. If you later collect real
shots with known outcomes (goal / no goal), that data can replace this formula with a fitted
logistic regression, which is what real xG models use.
"""
import argparse
import csv
import math

GOAL_WIDTH_M = 7.32  # official width of a goal mouth, post to post


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--ball", required=True, help="<video>_ball.csv from run.py")
    p.add_argument("--goals", required=True, help="<video>_goals.csv from run.py")
    p.add_argument("--fps", type=float, required=True, help="Video fps (printed by run.py / ffprobe)")
    p.add_argument("--out", default=None, help="Output CSV path (default: <ball csv>_shots.csv)")
    p.add_argument("--min-speed-percentile", type=float, default=90,
                    help="A frame counts as fast if its ball speed is above this percentile of all speeds")
    p.add_argument("--min-alignment", type=float, default=0.5,
                    help="cos(angle) between the ball's velocity and the ball->goal direction; "
                         "1.0 = moving straight at goal, 0.0 = moving sideways to it")
    p.add_argument("--merge-gap", type=int, default=8, help="Merge shot frames closer than this (frames)")
    p.add_argument("--lookback", type=int, default=6,
                    help="Frames to look back from the speed peak for the shot's origin/team")
    p.add_argument("--after-frames", type=int, default=20,
                    help="Frames after the shot to check whether the ball reached the goal box")
    p.add_argument("--max-distance-m", type=float, default=45,
                    help="Discard candidates farther than this from goal: a fast, roughly-toward-goal "
                         "ball movement from way outside the pitch is a pass caught by the alignment "
                         "check, not a shot, since real shots rarely come from this far")
    # hand-picked logistic coefficients: xG = sigmoid(a + b*distance_m + c*angle_deg)
    p.add_argument("--coef-a", type=float, default=-1.0)
    p.add_argument("--coef-dist", type=float, default=-0.10)
    p.add_argument("--coef-angle", type=float, default=0.05)
    return p.parse_args()


def read_ball_csv(path):
    rows = []
    with open(path, newline="") as f:
        for r in csv.DictReader(f):
            x = float(r["x"]) if r["x"] not in ("", None) else None
            y = float(r["y"]) if r["y"] not in ("", None) else None
            rows.append({"frame": int(r["frame"]), "x": x, "y": y, "team": r["team_possession"] or None})
    rows.sort(key=lambda r: r["frame"])
    return rows


def read_goal_csv(path):
    by_frame = {}
    with open(path, newline="") as f:
        for r in csv.DictReader(f):
            frame = int(r["frame"])
            row = {k: float(r[k]) for k in ("confidence", "x1", "y1", "x2", "y2")}
            if frame not in by_frame or row["confidence"] > by_frame[frame]["confidence"]:
                by_frame[frame] = row
    return by_frame


def fill_goal_per_frame(goal_by_frame, n_frames):
    """Forward/backward-fill the goal box so every frame has one (camera moves slowly frame to frame)."""
    known = sorted(goal_by_frame)
    if not known:
        return [None] * n_frames
    filled, last = [], None
    ki = 0
    for f in range(n_frames):
        while ki < len(known) and known[ki] <= f:
            last = goal_by_frame[known[ki]]
            ki += 1
        filled.append(last)
    # back-fill the very first frames, which had no goal detection yet
    first_known = goal_by_frame[known[0]]
    for f in range(len(filled)):
        if filled[f] is None:
            filled[f] = first_known
        else:
            break
    return filled


def percentile(values, p):
    s = sorted(values)
    if not s:
        return 0.0
    k = (len(s) - 1) * p / 100
    lo, hi = int(math.floor(k)), int(math.ceil(k))
    if lo == hi:
        return s[lo]
    return s[lo] + (s[hi] - s[lo]) * (k - lo)


def main():
    args = parse_args()
    ball = read_ball_csv(args.ball)
    n = ball[-1]["frame"] + 1 if ball else 0
    goal_by_frame = read_goal_csv(args.goals)
    goal_per_frame = fill_goal_per_frame(goal_by_frame, n)

    pos = {r["frame"]: (r["x"], r["y"]) for r in ball if r["x"] is not None}
    team = {r["frame"]: r["team"] for r in ball}

    # velocity + speed, only between frames that are close enough together (small tracking gaps are ok)
    speed = {}
    for f in range(1, n):
        if f in pos and (f - 1) in pos:
            x0, y0 = pos[f - 1]
            x1, y1 = pos[f]
            speed[f] = (x1 - x0, y1 - y0, math.hypot(x1 - x0, y1 - y0) * args.fps)

    if not speed:
        print("No consecutive ball detections found: cannot detect shots.")
        return

    speed_threshold = percentile([s[2] for s in speed.values()], args.min_speed_percentile)

    candidates = []
    for f, (vx, vy, sp) in speed.items():
        if sp < speed_threshold or f not in pos:
            continue
        goal = goal_per_frame[f]
        if goal is None:
            continue
        gx, gy = (goal["x1"] + goal["x2"]) / 2, (goal["y1"] + goal["y2"]) / 2
        bx, by = pos[f]
        to_goal = (gx - bx, gy - by)
        norm_v = math.hypot(vx, vy)
        norm_g = math.hypot(*to_goal)
        if norm_v == 0 or norm_g == 0:
            continue
        alignment = (vx * to_goal[0] + vy * to_goal[1]) / (norm_v * norm_g)
        if alignment >= args.min_alignment:
            candidates.append(f)

    if not candidates:
        print(f"No shot-like ball movement found (speed > {speed_threshold:.0f} px/s, "
              f"heading at the goal). Try lowering --min-speed-percentile or --min-alignment.")
        return

    # merge nearby candidate frames into shot events
    candidates.sort()
    events = [[candidates[0]]]
    for f in candidates[1:]:
        if f - events[-1][-1] <= args.merge_gap:
            events[-1].append(f)
        else:
            events.append([f])

    shots = []
    for ev in events:
        peak = max(ev, key=lambda f: speed[f][2])
        origin_f = max(peak - args.lookback, min(ev))
        while origin_f not in pos and origin_f < peak:
            origin_f += 1
        if origin_f not in pos:
            continue
        bx, by = pos[origin_f]

        shot_team = team.get(origin_f) or next(
            (team[f] for f in range(origin_f, max(origin_f - args.lookback, 0) - 1, -1) if team.get(f)), None
        )

        goal = goal_per_frame[peak]
        x1, y1, x2, y2 = goal["x1"], goal["y1"], goal["x2"], goal["y2"]
        goal_px_width = max(x2 - x1, 1.0)
        scale = GOAL_WIDTH_M / goal_px_width  # meters per pixel, calibrated off the goal box

        gcx, gcy = (x1 + x2) / 2, (y1 + y2) / 2
        dist_px = math.hypot(gcx - bx, gcy - by)
        distance_m = dist_px * scale

        gy_mid = (y1 + y2) / 2
        a = (x1 - bx, gy_mid - by)
        b = (x2 - bx, gy_mid - by)
        na, nb = math.hypot(*a), math.hypot(*b)
        angle_deg = 0.0
        if na > 0 and nb > 0:
            cos_a = max(-1.0, min(1.0, (a[0] * b[0] + a[1] * b[1]) / (na * nb)))
            angle_deg = math.degrees(math.acos(cos_a))

        z = args.coef_a + args.coef_dist * distance_m + args.coef_angle * angle_deg
        xg = 1 / (1 + math.exp(-z))

        if distance_m > args.max_distance_m:
            continue

        reached_goal = any(
            f in pos and x1 <= pos[f][0] <= x2 and y1 <= pos[f][1] <= y2
            for f in range(peak, min(peak + args.after_frames, n))
        )

        shots.append(
            dict(frame=origin_f, time_s=round(origin_f / args.fps, 1), team=shot_team or "",
                 x=round(bx, 1), y=round(by, 1), distance_m=round(distance_m, 1),
                 angle_deg=round(angle_deg, 1), xg=round(xg, 3),
                 reached_goal_box=reached_goal)
        )

    out_path = args.out or args.ball.replace(".csv", "") + "_shots.csv"
    if out_path == args.ball:
        out_path = args.ball[:-4] + "_shots.csv"
    with open(out_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["frame", "time_s", "team", "x", "y", "distance_m",
                                           "angle_deg", "xg", "reached_goal_box"])
        w.writeheader()
        w.writerows(shots)

    print(f"{len(shots)} shot(s) detected -> {out_path}\n")
    totals = {}
    for s in shots:
        t = s["team"] or "Unknown"
        totals.setdefault(t, {"shots": 0, "xg": 0.0, "reached": 0})
        totals[t]["shots"] += 1
        totals[t]["xg"] += s["xg"]
        totals[t]["reached"] += s["reached_goal_box"]
    print("===== xG summary (heuristic, see the file's docstring) =====")
    for t, v in totals.items():
        print(f"{t}: {v['shots']} shot(s), xG {v['xg']:.2f}, ball reached the goal box on {v['reached']}")


if __name__ == "__main__":
    main()

### `xg_overlay.py`

In [ ]:
%%writefile /content/soccer-video-analytics/xg_overlay.py
"""
xg_overlay.py - draws the shots + a running xG total onto the video run.py already produced
(put next to run.py). Runs AFTER compute_xg.py.

    python xg_overlay.py --video outputs/match_out.mp4 --shots outputs/match_ball_shots.csv \
        --teams outputs/match_teams.json --out outputs/match_out_xg.mp4

For every shot: a circle appears at the shot location for ~1 second with a small
"Shot | xG 0.12" label. A compact panel in the bottom-right accumulates each team's total xG as
shots happen, so it fills in over the course of the video, next to the video's own possession
board. See compute_xg.py's docstring for what the xG numbers do and don't account for.
"""
import argparse
import csv
import json
from functools import lru_cache

import numpy as np
import PIL.Image
import PIL.ImageDraw
import PIL.ImageFont

from viz_tweaks import HQWriter

FONT_PATH = "fonts/Gidole-Regular.ttf"
MARKER_COLOR = (0, 210, 255)  # BGR, like the rest of this codebase -> amber in the video
MARKER_SECONDS = 1.0
PANEL_W, PANEL_H = 300, 92  # slightly shorter than the possession board so it doesn't overlap on short videos
PANEL_MARGIN = 30
PANEL_BG = (20, 20, 24, 205)
PANEL_BORDER = (255, 255, 255, 45)
TEXT_MAIN = (255, 255, 255, 255)
TEXT_DIM = (176, 176, 184, 255)


@lru_cache(maxsize=8)
def _font(size):
    try:
        return PIL.ImageFont.truetype(FONT_PATH, size=size)
    except OSError:
        return PIL.ImageFont.load_default()


def _text(draw, xy, text, size, fill, anchor="l"):
    font = _font(size)
    l, t, r, b = draw.textbbox((0, 0), text, font=font)
    w, h = r - l, b - t
    x, y = xy
    if anchor == "m":
        x -= w / 2
    elif anchor == "r":
        x -= w
    draw.text((x - l, y - h / 2 - t), text, font=font, fill=fill)


def _rgb(color):
    return tuple(int(c) for c in color[:3])


def load_shots(path):
    with open(path, newline="") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r["frame"] = int(r["frame"])
        r["x"], r["y"] = float(r["x"]), float(r["y"])
        r["xg"] = float(r["xg"])
    rows.sort(key=lambda r: r["frame"])
    return rows


def load_teams(path):
    try:
        with open(path) as f:
            data = json.load(f)
        return {name: _rgb(t["color_bgr"]) + (255,) for name, t in data.items()}
    except (OSError, KeyError):
        return {}


def draw_marker(frame, shot, age_frames, total_frames):
    fade = max(0.0, 1.0 - age_frames / max(total_frames, 1))
    r = 14 + int(10 * (age_frames / max(total_frames, 1)))  # grows slightly as it fades
    x, y = shot["x"], shot["y"]
    overlay = PIL.Image.new("RGBA", frame.size, (0, 0, 0, 0))
    d = PIL.ImageDraw.Draw(overlay)
    alpha = int(255 * fade)
    d.ellipse([x - r, y - r, x + r, y + r], outline=MARKER_COLOR + (alpha,), width=3)
    label = f'Shot  \u00b7  xG {shot["xg"]:.2f}'
    lx, ly = x, max(y - r - 14, 16)
    bbox = d.textbbox((0, 0), label, font=_font(16))
    pad = 5
    d.rounded_rectangle(
        [lx - (bbox[2] - bbox[0]) / 2 - pad, ly - (bbox[3] - bbox[1]) / 2 - pad,
         lx + (bbox[2] - bbox[0]) / 2 + pad, ly + (bbox[3] - bbox[1]) / 2 + pad],
        radius=6, fill=(20, 20, 24, min(alpha, 205)),
    )
    _text(d, (lx, ly), label, 16, MARKER_COLOR + (alpha,), "m")
    return PIL.Image.alpha_composite(frame.convert("RGBA"), overlay).convert(frame.mode)


def draw_xg_panel(frame, totals, team_colors, origin=None):
    W, H = frame.size
    s = W / 1280.0
    pw, ph = int(PANEL_W * s), int(PANEL_H * s)
    margin = int(PANEL_MARGIN * s)
    # sits just below where the possession board goes (top-right), left of the frame edge
    x0, y0 = origin if origin else (W - pw - margin, margin + int(112 * s))

    img = PIL.Image.new("RGBA", (pw, ph), (0, 0, 0, 0))
    d = PIL.ImageDraw.Draw(img)
    d.rounded_rectangle([0, 0, pw - 1, ph - 1], radius=int(14 * s), fill=PANEL_BG, outline=PANEL_BORDER, width=1)
    _text(d, (pw / 2, 16 * s), "TOTAL xG", int(11 * s), TEXT_DIM, "m")

    names = list(totals.keys())
    row_y = 46 * s
    if len(names) == 0:
        _text(d, (pw / 2, row_y), "no shots yet", int(13 * s), TEXT_DIM, "m")
    else:
        col_w = pw / len(names)
        for i, name in enumerate(names):
            cx = col_w * (i + 0.5)
            color = team_colors.get(name, TEXT_MAIN)
            _text(d, (cx, row_y), name, int(12 * s), color, "m")
            _text(d, (cx, row_y + 26 * s), f"{totals[name]:.2f}", int(20 * s), TEXT_MAIN, "m")

    box = (x0, y0, x0 + pw, y0 + ph)
    region = frame.crop(box).convert("RGBA")
    region = PIL.Image.alpha_composite(region, img)
    frame.paste(region.convert(frame.mode), (x0, y0))
    return frame


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--video", required=True, help="The tracking+possession video from run.py (*_out.mp4)")
    ap.add_argument("--shots", required=True, help="*_ball_shots.csv from compute_xg.py")
    ap.add_argument("--teams", default=None, help="*_teams.json from run.py (for the real jersey colors)")
    ap.add_argument("--out", default=None, help="Output path (default: <video>_xg.mp4)")
    args = ap.parse_args()

    import cv2  # local import: keeps this file's own deps explicit even if cv2 patches waitKey elsewhere

    shots = load_shots(args.shots)
    team_colors = load_teams(args.teams) if args.teams else {}

    cap = cv2.VideoCapture(args.video)
    if not cap.isOpened():
        raise SystemExit(f"Could not open video: {args.video}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    marker_frames = max(int(round(MARKER_SECONDS * fps)), 1)

    out_path = args.out or args.video.rsplit(".", 1)[0] + "_xg.mp4"

    class _P:  # tiny stand-in so HQWriter can reuse its ffmpeg pipeline here
        def get_output_file_path(self):
            return out_path

    writer = HQWriter(_P(), fps)

    shots_by_frame = {}
    for s in shots:
        shots_by_frame.setdefault(s["frame"], []).append(s)

    totals = {}
    active = []  # shots currently being drawn: (shot, start_frame)
    i = 0
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        for s in shots_by_frame.get(i, []):
            team = s["team"] or "Unknown"
            totals[team] = totals.get(team, 0.0) + s["xg"]
            active.append((s, i))
        active = [(s, f0) for s, f0 in active if i - f0 < marker_frames]

        frame = PIL.Image.fromarray(frame_bgr)
        for s, f0 in active:
            frame = draw_marker(frame, s, i - f0, marker_frames)
        frame = draw_xg_panel(frame, totals, team_colors)

        writer.write(np.array(frame))
        i += 1
        if i % 100 == 0:
            print(f"{i}/{n_frames} frames")

    cap.release()
    writer.close()
    print(f"\nFinal totals: {', '.join(f'{k}: {v:.2f}' for k, v in totals.items()) or '(no shots)'}")


if __name__ == "__main__":
    main()

## 6. Settings

* `BALL_MODEL`: your ball detector (`.pt` trained with ultralytics).
* `PLAYER_MODEL`: optional. Leave it empty to use the pretrained YOLOv5x (COCO *person*). To use your own player detector,
  put its path here. Run once and read the line `[UltralyticsDetector] loaded ... | classes: {...}` to see its class names.
* `PLAYER_CLASSES`: optional, comma separated names to keep from `PLAYER_MODEL` (empty = every class except the ball).
* `GOAL_MODEL`: optional goal detector. When set, the goal boxes are drawn on the video (thin cyan box with the class name and confidence)
  and every detection is saved to `outputs/<video>_goals.csv` (frame, class, confidence, box, frame size).
* `COMPUTE_XG`: separate on/off switch for steps 10-11 (shot detection + xG). Keep it `False` to use the goal
  model just for the drawn boxes / CSV without computing xG.

In [ ]:
import os

BALL_MODEL     = "/content/drive/MyDrive/football_project/best.pt"
PLAYER_MODEL   = ""        # e.g. "/content/drive/MyDrive/football_project/player_best.pt"
PLAYER_CLASSES = ""        # e.g. "player,goalkeeper"
GOAL_MODEL     = ""        # e.g. "/content/drive/MyDrive/goal.pt"  (empty = no goal boxes drawn, steps 10-11 skipped)
COMPUTE_XG     = False     # True = also run steps 10-11 (shot detection + xG). Needs GOAL_MODEL set.

assert os.path.exists(BALL_MODEL), f"Ball model not found: {BALL_MODEL}"
assert not PLAYER_MODEL or os.path.exists(PLAYER_MODEL), f"Player model not found: {PLAYER_MODEL}"
assert not GOAL_MODEL or os.path.exists(GOAL_MODEL), f"Goal model not found: {GOAL_MODEL}"
assert not COMPUTE_XG or GOAL_MODEL, "COMPUTE_XG needs GOAL_MODEL set (distance/angle are measured against the goal box)"
print("Settings OK")

## 7. Choose the video

Run this cell and a **Choose files** button appears. Pick the video from your computer.
To use a video that is already on Google Drive, set `VIDEO_FROM_DRIVE` instead.

**To process another video, just run steps 7 and 8 again.**

In [ ]:
import os, re
from google.colab import files

VIDEO_FROM_DRIVE = ""      # e.g. "/content/drive/MyDrive/videos/match.mp4"  (empty = upload from your computer)

def clean_name(n):
    stem, ext = os.path.splitext(os.path.basename(n))
    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", stem).strip("_") or "video"
    return stem + (ext.lower() or ".mp4")

%cd /content/soccer-video-analytics
os.makedirs("videos", exist_ok=True)

if VIDEO_FROM_DRIVE:
    VIDEO_PATH = os.path.join("videos", clean_name(VIDEO_FROM_DRIVE))
    !cp "{VIDEO_FROM_DRIVE}" "{VIDEO_PATH}"
else:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was chosen. Run this cell again and pick a video.")
    name = next(iter(uploaded))
    VIDEO_PATH = os.path.join("videos", clean_name(name))
    with open(VIDEO_PATH, "wb") as f:
        f.write(uploaded[name])
    for n in uploaded:          # remove the copy Colab saved in the working folder
        if os.path.exists(n):
            os.remove(n)

print("Selected video:", VIDEO_PATH)

## 8. Run the tracking + possession

In [ ]:
%cd /content/soccer-video-analytics

extra_args = f'--player-model "{PLAYER_MODEL}"' if PLAYER_MODEL else ""
if PLAYER_MODEL and PLAYER_CLASSES:
    extra_args += f' --player-classes "{PLAYER_CLASSES}"'
if GOAL_MODEL:
    extra_args += f' --goal-model "{GOAL_MODEL}"'

!python run.py --video "{VIDEO_PATH}" --model "{BALL_MODEL}" {extra_args}

stem = os.path.basename(VIDEO_PATH).split(".")[0]
OUTPUT_PATH = os.path.join("outputs", stem + "_out.mp4")
print("\nOutput:", OUTPUT_PATH, f"({os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB)")
if GOAL_MODEL:
    print("Goal detections:", os.path.join("outputs", stem + "_goals.csv"))

## 9. Get the result

In [ ]:
import shutil
from google.colab import files

SAVE_TO_DRIVE_DIR = "/content/drive/MyDrive/football_project"   # "" to skip
DOWNLOAD          = True
SHOW_PREVIEW      = False    # embeds the video in the notebook (only for short clips)

if SAVE_TO_DRIVE_DIR:
    os.makedirs(SAVE_TO_DRIVE_DIR, exist_ok=True)
    shutil.copy(OUTPUT_PATH, SAVE_TO_DRIVE_DIR)
    goals_csv = OUTPUT_PATH.replace("_out.mp4", "_goals.csv")
    if os.path.exists(goals_csv):
        shutil.copy(goals_csv, SAVE_TO_DRIVE_DIR)
    print("Copied to", SAVE_TO_DRIVE_DIR)
if SHOW_PREVIEW:
    from IPython.display import Video
    display(Video(OUTPUT_PATH, embed=True, width=800))
if DOWNLOAD:
    files.download(OUTPUT_PATH)

## 10. Shots + xG (optional)

Detects shots from the ball's speed and direction (no training needed) and estimates a rough,
heuristic xG per shot — see the docstring in `compute_xg.py` for what it does and does not account for.
Needs `--goal-model` to have been set in step 6 (the ball CSV alone isn't enough: the goal position is
what distance and angle are measured against).

In [ ]:
if not COMPUTE_XG:
    print("Set COMPUTE_XG = True in step 6 (and GOAL_MODEL) and re-run steps 7-8 to enable this step.")
else:
    ball_csv = os.path.join("outputs", stem + "_ball.csv")
    goals_csv = os.path.join("outputs", stem + "_goals.csv")
    fps_str = !ffprobe -v error -select_streams v:0 -show_entries stream=r_frame_rate -of csv=p=0 "{VIDEO_PATH}"
    num, den = fps_str[0].split("/")
    fps = float(num) / float(den)

    !python compute_xg.py --ball "{ball_csv}" --goals "{goals_csv}" --fps {fps}

    shots_csv = ball_csv.replace("_ball.csv", "_ball_shots.csv")
    if SAVE_TO_DRIVE_DIR and os.path.exists(shots_csv):
        shutil.copy(shots_csv, SAVE_TO_DRIVE_DIR)
        print("Copied to", SAVE_TO_DRIVE_DIR)

## 11. Draw the shots + xG on the video (optional)

Takes the video from step 8 and the shots from step 10, and draws a marker on each shot plus a
running "Total xG" panel that fills in as the video plays. Needs step 10 to have run first.

In [ ]:
if not COMPUTE_XG:
    print("Set COMPUTE_XG = True in step 6 and re-run steps 7-8-10 to enable this step.")
else:
    shots_csv = os.path.join("outputs", stem + "_ball_shots.csv")
    teams_json = os.path.join("outputs", stem + "_teams.json")
    XG_VIDEO_PATH = os.path.join("outputs", stem + "_out_xg.mp4")

    !python xg_overlay.py --video "{OUTPUT_PATH}" --shots "{shots_csv}" --teams "{teams_json}" --out "{XG_VIDEO_PATH}"

    if SAVE_TO_DRIVE_DIR:
        shutil.copy(XG_VIDEO_PATH, SAVE_TO_DRIVE_DIR)
        print("Copied to", SAVE_TO_DRIVE_DIR)
    if DOWNLOAD:
        files.download(XG_VIDEO_PATH)

---
**Notes**
* Before committing this notebook to GitHub: `Edit > Clear all outputs`.
* Model weights and videos are not part of the code: keep them out of the repository (or use Git LFS / GitHub Releases).
* The base pipeline is by [Tryolabs](https://github.com/tryolabs/soccer-video-analytics) (MIT). `ultralytics` is AGPL-3.0, keep that in mind when publishing.